# 🏦 Банковский счёт: `Account` и `CreditAccount`

> **Цель:** реализовать класс `Account`, моделирующий банковский счёт, с историей операций и аналитикой.  
> Затем создать наследника `CreditAccount`.

---

## 📌 1. Класс `Account`

### Инициализация

```python
Account(account_holder, balance=0)
```

| Параметр | Тип | По умолчанию | Описание |
|----------|-----|--------------|----------|
| `account_holder` | `str` | — | Имя владельца счёта |
| `balance` | `float` | `0` | Начальный баланс. Не может быть отрицательным |

### Атрибуты

| Атрибут | Тип | Описание |
|---------|-----|----------|
| `holder` | `str` | Имя владельца |
| `_balance` | `float` | Приватный текущий баланс |
| `operations_history` | `list` / `tuple` | История операций |

> ⚠️ **Важно:** каждая операция хранится как структурированная запись — словарь или кортеж.  
> Минимальный набор полей:
> - тип операции: `'deposit'` или `'withdraw'`;
> - сумма;
> - дата и время;
> - текущий баланс после операции;
> - статус: `'success'` или `'fail'`.

#### Пример записи операции

```python
{
    "type": "deposit",
    "amount": 1000.0,
    "datetime": datetime.now(),
    "balance_after": 1000.0,
    "status": "success"
}
```

---

## 🛠️ 2. Методы `Account`

### `__init__(self, account_holder, balance=0)`
Конструктор класса.

### `deposit(self, amount)`
Пополнение счёта.

- принимает сумму;
- сумма должна быть положительной;
- при успехе обновляет баланс и добавляет запись в историю.

### `withdraw(self, amount)`
Снятие средств.

- принимает сумму;
- сумма должна быть положительной;
- проверяет, достаточно ли средств;
- если средств недостаточно — операция не проходит, но попытка фиксируется в истории со статусом `'fail'`;
- при успехе обновляет баланс и добавляет запись.

### `get_balance(self)`
Возвращает текущий баланс.

### `get_history(self)`
Возвращает историю операций.

> 💡 **Подсказка:** для работы с датой и временем используйте `datetime.now()` из модуля `datetime`.

---

## 💳 3. Класс `CreditAccount(Account)`

Наследует всю функциональность `Account` и добавляет кредитную логику.

### Особенности

- При инициализации принимает дополнительный параметр `credit_limit`.
- Баланс может быть отрицательным, но не ниже `-credit_limit`.
- По запросу показывает доступные кредитные средства:  
  `доступно = текущий баланс + credit_limit`
- В историю операций добавляется информация о том, были ли использованы кредитные средства.

### Инициализация

```python
CreditAccount(account_holder, balance=0, credit_limit=0)
```

---

## ✅ Чек-лист реализации

- [ ] Создан класс `Account`.
- [ ] Реализован конструктор с проверкой баланса.
- [ ] Реализованы `deposit`, `withdraw`, `get_balance`, `get_history`.
- [ ] История операций хранит структурированные записи.
- [ ] Неудачные попытки снятия фиксируются.
- [ ] Создан класс `CreditAccount`.
- [ ] Учтён кредитный лимит.
- [ ] Добавлена информация об использовании кредитных средств.

---

## 📚 Полезные импорты

```python
from datetime import datetime
```

---

> **Совет:** для красивого вывода истории можно вернуть список словарей или преобразовать его в `pandas.DataFrame`.

# 📊 Критерии оценивания проекта

> **Максимум по базовым критериям: 8 баллов** — это оценка **«отлично»**.  
> 💡 Преподаватель может добавить **1–2 балла** и поставить **9 или 10**, если работа выполнена сверх ожиданий и выходит за рамки программы.

---

| № | Критерий | Баллы | Описание |
|:--:|:---------|:-----:|:---------|
| 1 | 🧱 **Корректность ООП** | **3** | Класс корректно инициализируется, используются инкапсуляция (приватные атрибуты, геттеры), методы `deposit` и `withdraw` работают без ошибок, логически верно обрабатывают неверные входные данные (отрицательные суммы) |
| 2 | 📜 **История операций** | **2** | История сохраняется в структурированном виде. Метод `get_history` возвращает информацию в удобном формате |
| 3 | 🕒 **Работа с датами** | **1** | В каждой операции корректно фиксируется дата и время её проведения с помощью модуля `datetime` |
| 4 | 🧬 **Наследование** | **2** | Корректно реализована функциональность класса `CreditAccount(Account)`: в реализации учтены все заданные особенности кредитного счёта |
| 5 | ✨ **Чистота кода** | **−1** за каждое нарушение | Код откомментирован, читаемый, соответствует PEP8 |

---

> **Итого:** `3 + 2 + 1 + 2 = 8` баллов — оценка **«отлично»**.  
> 🚀 Дополнительные баллы: **9–10**, если работа выходит за рамки программы.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_EVEN
from enum import StrEnum


ZERO_MONEY = Decimal("0.00")
MONEY_QUANTUM = Decimal("0.01")  # Задает два знака после разделителя


def normalize_money(value: object) -> Decimal:
    """Проверяет и округляет денежное значение до двух знаков."""

    if not isinstance(value, Decimal):
        raise TypeError("Денежное значение должно иметь тип Decimal")

    if not value.is_finite():
        raise ValueError("Денежное значение должно быть конечным числом")

    return value.quantize(
        MONEY_QUANTUM,
        rounding=ROUND_HALF_EVEN,
    )

class OperationType(StrEnum):
    """Определяет тип банковской операции."""

    DEPOSIT = "deposit"
    WITHDRAW = "withdraw"


class OperationStatus(StrEnum):
    """Определяет результат банковской операции."""

    SUCCESS = "success"
    FAIL = "fail"



@dataclass(frozen=True, slots=True)
class Transaction:
    """Представляет неизменяемую банковскую операцию."""

    operation_type: OperationType
    amount: Decimal
    created_at: datetime
    balance_after: Decimal
    status: OperationStatus
    credit_used: bool | None = None


class Account:
    """Моделирует банковский счёт и хранит историю операций."""

    def __init__(self, account_holder: str, balance: Decimal = ZERO_MONEY) -> None:
        """Создаёт банковский счёт с указанным владельцем и начальным балансом."""

        self.holder: str = account_holder
        balance = normalize_money(balance)

        if balance < ZERO_MONEY:
            raise ValueError("Начальный баланс не может быть отрицательным")

        self._balance: Decimal = balance
        self._operations_history: list[Transaction] = []

    def deposit(self, amount: Decimal) -> bool:
        """Пополняет счёт на положительную сумму."""

        amount = normalize_money(amount)

        if amount <= ZERO_MONEY:
            raise ValueError("Сумма пополнения должна быть положительной")

        self._balance += amount

        self._add_history(
            operation_type=OperationType.DEPOSIT,
            amount=amount,
            date_time=datetime.now(timezone.utc),
            balance_after=self._balance,
            status=OperationStatus.SUCCESS,
        )
        return True

    def withdraw(self, amount: Decimal) -> bool:
        """Снимает средства при наличии достаточной суммы на счёте."""

        amount = normalize_money(amount)

        if amount <= ZERO_MONEY:
            raise ValueError("Сумма снятия должна быть положительной")

        if amount > self._balance:
            status = OperationStatus.FAIL
        else:
            self._balance -= amount
            status = OperationStatus.SUCCESS
        self._add_history(
            operation_type=OperationType.WITHDRAW,
            amount=amount,
            date_time=datetime.now(timezone.utc),
            balance_after=self._balance,
            status=status,
        )
        return status == OperationStatus.SUCCESS

    def get_balance(self) -> Decimal:
        """Возвращает текущий баланс счёта."""

        return self._balance

    def _add_history(
        self,
        operation_type: OperationType,
        amount: Decimal,
        date_time: datetime,
        balance_after: Decimal,
        status: OperationStatus,
        credit_used: bool | None = None,
    ) -> None:
        """Добавляет запись об операции в историю счёта."""

        transaction = Transaction(
            operation_type=operation_type,
            amount=amount,
            created_at=date_time,
            balance_after=balance_after,
            status=status,
            credit_used=credit_used,
        )

        self._operations_history.append(transaction)

    def get_history(self) -> tuple[Transaction, ...]:
        """Возвращает неизменяемую историю операций."""

        return tuple(self._operations_history)

    def __str__(self) -> str:
        """Возвращает строковое представление банковского счёта."""

        return f"Владелец: {self.holder}. Баланс: {self._balance}"


class CreditAccount(Account):
    """Моделирует банковский счёт с кредитным лимитом."""

    def __init__(
        self,
        account_holder: str,
        balance: Decimal = ZERO_MONEY,
        credit_limit: Decimal = ZERO_MONEY,
    ) -> None:
        """Создаёт банковский счёт с балансом и кредитным лимитом."""

        super().__init__(account_holder, balance)

        credit_limit = normalize_money(credit_limit)

        if credit_limit < ZERO_MONEY:
            raise ValueError("Кредитный лимит не может быть отрицательным")

        self._credit_limit: Decimal = credit_limit

    def withdraw(self, amount: Decimal) -> bool:
        """Снимает средства с учётом установленного кредитного лимита."""

        amount = normalize_money(amount)
        available_funds = self.get_available_funds()

        if amount <= ZERO_MONEY:
            raise ValueError("Сумма снятия должна быть положительной")

        if amount > available_funds:
            status = OperationStatus.FAIL
        else:
            self._balance -= amount
            status = OperationStatus.SUCCESS

        credit_used = status == OperationStatus.SUCCESS and self._balance < ZERO_MONEY
        self._add_history(
            operation_type=OperationType.WITHDRAW,
            amount=amount,
            date_time=datetime.now(timezone.utc),
            balance_after=self._balance,
            status=status,
            credit_used=credit_used,
        )

        return status == "success"

    def get_available_funds(self) -> Decimal:
        """Возвращает общую сумму доступных собственных и кредитных средств."""

        return self._balance + self._credit_limit

    def get_credit_limit(self) -> Decimal:
        """Возвращает кредитный лимит."""

        return self._credit_limit

    def __str__(self) -> str:
        """Возвращает строковое представление кредитного счёта."""

        available_funds = self.get_available_funds()

        return (
            f"Владелец: {self.holder}. "
            f"Баланс: {self._balance}. "
            f"Кредитный лимит: {self._credit_limit}. "
            f"Доступно средств: {available_funds}"
        )

In [ ]:
account = Account(
    "Илья",
    Decimal("1000.00"),
)

account.deposit(Decimal("500.30"))

history = account.get_history()
transaction = history[0]

print(transaction.operation_type)
print(transaction.amount)
print(transaction.created_at)
print(transaction.balance_after)
print(transaction.status)
print(transaction.credit_used)

credit_account = CreditAccount(
    "Анна",
    Decimal("1000.00"),
    Decimal("5000.00"),
)

credit_account.withdraw(Decimal("3000.00"))

transaction = credit_account.get_history()[0]

print(transaction.credit_used)

deposit
500.30
2026-10-09 07:01:34.089158+00:00
1500.30
success
None
True


AttributeError: 'tuple' object has no attribute 'append'